# Web Search Ingestion for RAG

The lifecycle this notebook walks through:

```
query -> SEARCH (Tavily/mock) -> FETCH (httpx) -> PARSE (baseline/trafilatura/docling/unstructured)
      -> CLEAN -> CHUNK -> documents.jsonl / chunks.jsonl (ready for embedding)
```

Every cell below runs **offline** against bundled fixture pages, so it works
without a Tavily key. Set `TAVILY_API_KEY` in `.env` to switch to live search.


In [1]:
from pathlib import Path

from web_ingestion.config import get_settings, FIXTURES_DIR
from web_ingestion.models import FetchedDocument
from web_ingestion.parsers import available_parsers, get_parser, unavailable_reason
from web_ingestion.transform.clean import clean_text
from web_ingestion.transform.chunk import chunk_text

settings = get_settings()
print("mock mode:", settings.mock_mode, "| default parser:", settings.default_parser)

mock mode: True | default parser: baseline


## 1. Which parsers can you use?

`baseline` is bundled. `trafilatura`, `docling` and `unstructured` are optional
extras than need `uv sync --extra <name>`.

In [2]:
for name, ok in available_parsers().items():
    print(f"{name:14s} {'installed' if ok else 'NOT installed -- ' + unavailable_reason(name)}")

baseline       installed
trafilatura    installed
docling        NOT installed -- docling is not installed -> run: uv sync --extra docling  (heavy: downloads models)
unstructured   installed


## 2. Fetch a page (offline fixture)

In mock mode `HttpFetcher` reads `fixtures/pages/*.html` instead of hitting the
network. In live mode it downloads real pages with the same code path.

In [3]:
raw = (FIXTURES_DIR / "pages" / "rag-chunking.html").read_bytes()
doc = FetchedDocument(
    url="https://example.com/docs/rag-chunking",
    content=raw,
    content_type="text/html",
    title_hint="Chunking strategies for RAG systems",
)
print("fetched bytes:", len(doc.content), "| is_pdf:", doc.is_pdf)

fetched bytes: 3308 | is_pdf: False


## 3. Parse with every installed parser

The same document is parsed by each installed parser; we compare how much text
comes out, how much survives cleaning, the parse time, and what structure items
(headings) were kept.

In [4]:
results = {}
for name, ok in available_parsers().items():
    if not ok:
        continue
    parser = get_parser(name)
    parsed = parser.parse(doc)
    cleaned = clean_text(parsed.text)
    results[name] = {"parsed": parsed, "cleaned_chars": len(cleaned)}
    print(
        f"{name:12s} raw={len(parsed.text):>7}  cleaned={len(cleaned):>7}  "
        f"structure={len(parsed.structure):>3}  title={parsed.title!r}\n"
        f"{' ':12s} metadata={list(parsed.metadata.items())}"
    )

baseline     raw=   1797  cleaned=   1797  structure=  6  title='Chunking strategies for RAG systems | Acme Docs'
             metadata=[('description', 'Fixed-size, sentence-aware and semantic chunking compared for retrieval quality.')]
trafilatura  raw=   1814  cleaned=   1814  structure=  6  title='Chunking strategies for RAG systems'
             metadata=[('title', 'Chunking strategies for RAG systems'), ('url', 'https://example.com/docs/rag-chunking'), ('hostname', 'example.com'), ('description', 'Fixed-size, sentence-aware and semantic chunking compared for retrieval quality.'), ('sitename', 'example.com'), ('date', '2026-01-01'), ('raw_text', 'Chunking strategies for RAG systems\nChunking is the step where a clean document becomes retrieval units. Get it wrong and even a strong embedding model cannot recover: a chunk that cuts a definition in half will not match the query it should answer.\nFixed-size windows\nThe simplest approach is to cut text into windows of N characters or

### Raw vs cleaned preview
Pick the parser you want to inspect and look at the start of the text before and
after cleaning.

In [5]:
for name, entry in results.items():
    parsed = entry["parsed"]
    cleaned = clean_text(parsed.text)
    print("=" * 70)
    print(f"Parser: {name}   ({len(parsed.text)} -> {len(cleaned)} chars)")
    print("=" * 70)
    print(parsed.text[:600])
    print("\n--- cleaned ---\n")
    print(cleaned[:600])
    print()

Parser: baseline   (1797 -> 1797 chars)
Chunking strategies for RAG systems

Chunking is the step where a clean document becomes retrieval units. Get it wrong
and even a strong embedding model cannot recover: a chunk that cuts a definition in
half will not match the query it should answer.

Fixed-size windows

The simplest approach is to cut text into windows of N characters or tokens with a
sliding overlap. It is cheap, deterministic and surprisingly competitive when the
source documents already have short, self-contained paragraphs.

The failure mode is structural: a window can start in the middle of a sentence and
end in the midd

--- cleaned ---

Chunking strategies for RAG systems
 Chunking is the step where a clean document becomes retrieval units. Get it wrong and even a strong embedding model cannot recover: a chunk that cuts a definition in half will not match the query it should answer.
 Fixed-size windows
 The simplest approach is to cut text into windows of N characters or 

## 4. Cleaning rules

Cleaning is mostly subtraction: normalize unicode, collapse wrapped lines,
drop boilerplate lines (cookie banners, "All rights reserved", newsletter
teasers, repeated nav items), then dedupe repeated lines. Test the rules directly:

In [6]:
messy = (
    "Building a web ingestion pipeline\n"
    "Accept all cookies to continue.\n"
    "Real paragraph with   weird   spacing and a soft\n"
    "wrapped line across two rows.\n"
    "Subscribe to our newsletter\n"
    "All  rights  reserved.  Copyright(c) 2026.\n"
    "Real paragraph with   weird   spacing and a soft\n"
    "wrapped line across two rows."
)
print("BEFORE:")
print(messy)
print("\nAFTER clean_text:")
print(clean_text(messy))

BEFORE:
Building a web ingestion pipeline
Accept all cookies to continue.
Real paragraph with   weird   spacing and a soft
wrapped line across two rows.
Subscribe to our newsletter
All  rights  reserved.  Copyright(c) 2026.
Real paragraph with   weird   spacing and a soft
wrapped line across two rows.

AFTER clean_text:
Building a web ingestion pipeline Real paragraph with weird spacing and a soft wrapped line across two rows.
Real paragraph with weird spacing and a soft wrapped line across two rows.


## 5. Chunking

Paragraph-aware character windows with overlap: each chunk stays below the size
budget, and the tail of chunk N is repeated at the head of chunk N+1 so facts
straddling a boundary stay findable.

In [7]:
text = clean_text(results["baseline"]["parsed"].text)
chunks = chunk_text(text, doc_id="rag-chunking", url=doc.url, title=parsed.title,
                    chunk_size=settings.chunk_size, overlap=settings.chunk_overlap)
print(f"{len(chunks)} chunks of {len(text)} chars (size={settings.chunk_size}, overlap={settings.chunk_overlap})")
for c in chunks:
    print(f"-- {c.id}: {c.char_count} chars, ~{c.approx_tokens} tokens")
    print(c.text[:180].replace('\n', ' / '))
    print(c.text[-60:].replace('\n', ' / '), "  << TAIL of prev chunk repeats here\n")

2 chunks of 1797 chars (size=1200, overlap=200)
-- rag-chunking::0: 1200 chars, ~300 tokens
Chunking strategies for RAG systems /  Chunking is the step where a clean document becomes retrieval units. Get it wrong and even a strong embedding model cannot recover: a chunk tha
undary. It costs an extra embedding pass over the corpus and   << TAIL of prev chunk repeats here

-- rag-chunking::1: 797 chars, ~199 tokens
emantic approaches embed consecutive sentences and split where the embedding similarity drops, which means a topic change becomes a chunk boundary. It costs an extra embedding pass
downstream answer quality far more than prompt wording does.   << TAIL of prev chunk repeats here



## 6. Full pipeline in one call

`run_pipeline` does everything (search the mock fixture set, fetch, parse, clean,
chunk) and writes `data/runs/<timestamp>/` with per-document JSONL ready to be
embedded and indexed.

In [8]:
from web_ingestion.pipeline import run_pipeline

result = run_pipeline("attention is all you need transformer architecture explained", settings=settings)
print(f"run {result.run_id}: {len(result.records)} docs, {result.total_chunks} chunks, errors={len(result.errors)}")
print(f"outputs: {result.run_dir}")
print((result.run_dir / "report.md").read_text(encoding="utf-8"))

run 20261007-135419: 3 docs, 7 chunks, errors=0
outputs: /Users/Denis/Desktop/AI-SEPTEMBER/web-search-ingestion/data/runs/20261007-135419
# Run 20261007-135419

- Query: `attention is all you need transformer architecture explained`
- Provider: mock (mock=True)
- Parser: baseline
- Documents: 3 / 3
- Chunks: 7
- Errors: 0

| URL | raw chars | cleaned chars | chunks |
|---|---:|---:|---:|
| https://example.com/blog/attention-is-all-you-need | 2022 | 2022 | 2 |
| https://example.com/docs/rag-chunking | 1797 | 1797 | 2 |
| https://example.com/guides/web-ingestion-pipeline | 2257 | 2257 | 3 |



## 7. Going live with Tavily

1. `cp .env.example .env` and set `TAVILY_API_KEY=...` (https://app.tavily.com)
2. Re-run this notebook: `settings.mock_mode` flips to `False`, `run_pipeline`
   and `web-ingest search "query"` will hit the real API and download actual
   pages instead of fixtures.

The only code that changes is the search client and the fetcher's URL
resolution — the parse/clean/chunk stages are identical.